## NLP Feature Extraction Pipeline
This notebook loads the raw, uncleaned scraped data and processes the text fields to generate lightweight NLP features for the Streamlit dashboard.

In [1]:
!pip install textstat
!pip install vaderSentiment

In [1]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
from sklearn.linear_model import Ridge
import numpy as np
from transformers import pipeline
import textstat
import sys
import os
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sys.path.append(os.path.abspath(".."))
from processing.functions import *

data_path = "../data/cars_and_bids_full_history_v3.csv"
cols_to_keep = ['URL', 'Make', 'Model', 'Sold_Price', 'Modifications', 'Equipment', 'Highlights', 'Known Flaws', 'Recent Service History', 'Ownership History', 'Other Items Included in Sale', 'Seller Notes']
df = pd.read_csv(data_path, usecols=cols_to_keep)

# Quick cleaning of Model and Sold_Price columns
df['Sold_Price'] = df['Sold_Price'].apply(clean_currency)
df = df.dropna(subset=['Sold_Price'])
df['Model'] = df['Model'].apply(clean_model)

pd.set_option('display.max_columns', None)

df.head()

2026-03-19 04:13:16.031327: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,URL,Sold_Price,Make,Model,Highlights,Equipment,Modifications,Known Flaws,Recent Service History,Ownership History,Other Items Included in Sale,Seller Notes
0,https://carsandbids.com/auctions/rkdqRPJN/2026...,226000.0,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",A build sheet is provided in the photo gallery...,NaN,NaN,The seller states that this G63 has not requir...,The seller is representing this G63 on behalf ...,2 keys Owner's manuals,"There is a loan on this vehicle, and sale proc..."
1,https://carsandbids.com/auctions/30Op4L4g/2026...,76500.0,BMW,G8X M4,"THIS... is a 2026 BMW M4 Coupe, finished in Fr...","A build sheet is provided in the gallery, and ...",NaN,NaN,The attached Carfax history report shows that ...,The seller reportedly purchased this M4 new in...,2 keys Owner's manual Window sticker,The seller states that clear paint protection ...
2,https://carsandbids.com/auctions/3010V6BV/2026...,215485.0,Mercedes-Benz,G Wagen,"THIS… is a 2026 Mercedes-AMG G63, finished in ...",A build sheet is provided in the photo gallery...,NaN,NaN,The seller states that this G63 has not requir...,The seller reportedly purchased this G63 new i...,2 keys Owner's manual Build sheet All-weather ...,"There is a loan on this vehicle, and sale proc..."
3,https://carsandbids.com/auctions/KZB8oVXk/2026...,356000.0,Porsche,992 911,"THIS... is a 2026 Porsche 911 GT3 Touring, fin...",A build sheet is provided in the photo gallery...,NaN,NaN,The selling dealer states that this 911 has no...,The selling dealer reportedly acquired this 91...,2 keys Owner's manual Build sheet,NaN
7,https://carsandbids.com/auctions/9lDZOmLz/2026...,66000.0,Rivian,R1S,"THIS… is a 2026 Rivian R1S Adventure Edition, ...",A partial list of notable factory equipment re...,NaN,NaN,The seller states that this Rivian has not req...,The seller reportedly purchased this Rivian ne...,2 key cards Safety Guide All-weather floor mat...,"There is a loan on this vehicle, and sale proc..."


### 1. Entity Recognition: Premium Aftermarket Brands
Extracts specific high-value brands from the `Modifications` column.


In [3]:
brands = [
    "bilstein", "ohlins", "bbs", "recaro", "dinan", 
    "kw", "michelin", "koni", "brembo", "borla", "bosch",
    "apr", "momo", "nardi", "holley", "edgemotorsport", "moog", "acdelco", "hks"
]

def extract_brands(text):
    if pd.isna(text) or not isinstance(text, str):
        return []

    text_lower = text.lower()
    found_brands = []

    for brand in brands:
        if re.search(rf'\b{brand}\b', text_lower):
            found_brands.append(brand.capitalize())

    return found_brands

# Combine relevant fields
df['Combined_Mods'] = df['Modifications'].fillna("") + " " + \
                      df['Equipment'].fillna("") + " " + \
                      df.get('Other Items Included in Sale', pd.Series("")).fillna("")
                      
df['Extracted_Brands'] = df['Combined_Mods'].apply(extract_brands)
df['Has_Premium_Mods'] = df['Extracted_Brands'].apply(lambda x: 1 if len(x) > 0 else 0)
df['Premium_Brand_Count'] = df['Extracted_Brands'].apply(len)
df['Extracted_Brands_List'] = df['Extracted_Brands'].apply(lambda x: ", ".join(x))

# Preview listings that actually have premium mods
display(df[df['Has_Premium_Mods'] == 1][['Make', 'Model', 'Modifications', 'Equipment', 'Other Items Included in Sale', 'Extracted_Brands_List']].head())


,Make,Model,Modifications,Equipment,Other Items Included in Sale,Extracted_Brands_List
34,Subaru,BRZ,NaN,A partial list of notable equipment reported b...,2 keys Owner's manual Rubber floor and trunk mats,Brembo
37,Lexus,IS,NaN,A build sheet is provided in the photo gallery...,2 keys 2 key holders Owner's manual Lexus of Q...,Bbs
39,Toyota,Tacoma,NaN,A build sheet is provided in the photo gallery...,2 keys Owner's manual Booklets,Bilstein
41,BMW,G80 M3,Notable modifications reported by the seller i...,A window sticker is provided in the photo gall...,2 keys Owner's manual Window sticker All-weath...,Michelin
45,Subaru,BRZ,Notable modifications reported by the seller i...,A Monroney Label is provided in the photo gall...,2 key fobs Window sticker Set of winter tires ...,Brembo


### 2. Topic Modeling: Listing Archetypes
Uses Non-Negative Matrix Factorization (NMF) to cluster cars based on the combined text of their Highlights and Recent Service history.

In [4]:
cols_for_topics = ['Highlights', 'Equipment', 'Modifications', 'Seller Notes']
if all(col in df.columns for col in cols_for_topics):
    print("Combining text and fitting NMF model...")
    
    text_data = df['Highlights'].fillna("") + " " + \
                df['Equipment'].fillna("") + " " + \
                df['Modifications'].fillna("") + " " + \
                df['Seller Notes'].fillna("")
    text_list = text_data.tolist()
    
    # 1. Base auction boilerplate stop words
    auction_stop_words = [
        'seller', 'reports', 'states', 'gallery', 'pictured', 'known', 'flaws', 
        'finished', 'power', 'provided', 'carfax', 'report', 'shows', 'notable', 
        'equipment', 'includes', 'included', 'sale', 'history', 'attached', 
        'minor', 'damage', 'vehicle', 'additionally', 'purchase', 'pre', 
        'inspection', 'owner', 'manual', 'keys', 'car', 'auction', 'build', 'sheet',
        'package', 'bose', 'trim', 'windows', 'air', 'conditioning', 'seats', 'heated',
        'reported', 'dealer', 'include', 'performed', 'title', 'lender', 'sticker', 'driving',
        'loan', 'plus', 'selling', '2022', 'road'
    ]
    
    # 2. DYNAMIC BRAND STOP WORDS: Extract all makes and models from the dataframe
    makes = df['Make'].dropna().str.lower().unique().tolist()
    # Models are often multi-word (e.g. "911 Carrera"), so we split and explode them
    models_raw = df['Model'].dropna().str.lower().str.split().explode().unique().tolist()
    
    # Clean up any stray punctuation in the model names
    models = [re.sub(r'[^a-z0-9]', '', str(m)) for m in models_raw if str(m).strip()]
    
    # Add common trim/brand jargon that might not be exactly in the Make/Model columns
    extra_brands = ['amg', 'benz', 'carrera', 'm3', 'm4', 'boxster', 'cayman', 'cayenne', 'macan']
    
    # Combine all stop words into one master list
    custom_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words()) + \
                        auction_stop_words + makes + models + extra_brands
                        
    # Ensure custom stop words are unique
    custom_stop_words = list(set(custom_stop_words))
    
    # 3. Vectorize with the hyper-aggressive stop word list
    tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, stop_words=custom_stop_words)
    tfidf_matrix = tfidf_vectorizer.fit_transform(text_list)
    
    N_TOPICS = 4
    nmf_model = NMF(n_components=N_TOPICS, random_state=42, init='nndsvda')
    nmf_features = nmf_model.fit_transform(tfidf_matrix)
    
    df['Archetype_Cluster'] = nmf_features.argmax(axis=1)
    
    feature_names = tfidf_vectorizer.get_feature_names_out()
    print("\n--- Cluster Keywords ---")
    for topic_idx, topic in enumerate(nmf_model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-10 - 1:-1]]
        print(f"Cluster {topic_idx}: {', '.join(top_words)}")

Combining text and fitting NMF model...


/opt/conda/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['alfa', 'aston', 'bae', 'cars', 'company', 'cruise', 'general', 'jurassic', 'little', 'local', 'manx', 'martin', 'mercedes', 'meyers', 'military', 'motors', 'roadsters', 'rolls', 'romeo', 'royce', 'stevenson', 'stewart', 'suspensions', 'systems', 'unlimited'] not in stop_words.
  warnings.warn(



--- Cluster Keywords ---
Cluster 0: control, leather, automatic, adjustable, soft, climate, upholstery, sunroof, 18, sound
Cluster 1: aftermarket, cloth, unit, head, engine, imported, state, japanese, kit, 15
Cluster 2: carbon, fiber, performance, black, active, 19, assist, adaptive, exhaust, 20
Cluster 3: row, electric, pack, battery, ventilated, elevation, compressor, edition, motors, lithium


/opt/conda/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1728: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


## 3. Seller Transparency and "Effort" Scoring
Calculates word count, structural density (bullet points), readability, and sentiment across the main text fields to gauge seller effort.

In [5]:
def get_word_count(text):
    if pd.isna(text) or not isinstance(text, str) or len(text.strip()) == 0:
        return 0
    return textstat.lexicon_count(text, removepunct=True)

# Map the raw column names to clean, code-friendly column names
text_cols_map = {
    'Modifications': 'Modifications_WC',
    'Equipment': 'Equipment_WC',
    'Highlights': 'Highlights_WC',
    'Known Flaws': 'Known_Flaws_WC',
    'Recent Service History': 'Recent_Service_WC',
    'Ownership History': 'Ownership_WC',
    'Other Items Included in Sale': 'Other_Items_WC',
    'Seller Notes': 'Seller_Notes_WC'
}

if all(col in df.columns for col in text_cols_map.keys()):
    print("Calculating Section-Specific Word Counts...")
    
    # Iterate through the dictionary and apply the word count function
    for orig_col, new_col in text_cols_map.items():
        df[new_col] = df[orig_col].apply(get_word_count)
        
    display(df[['Make', 'Model'] + list(text_cols_map.values())].head())
else:
    print("Missing required columns for section-specific analysis.")

Calculating Section-Specific Word Counts...


,Make,Model,Modifications_WC,Equipment_WC,Highlights_WC,Known_Flaws_WC,Recent_Service_WC,Ownership_WC,Other_Items_WC,Seller_Notes_WC
0,Mercedes-Benz,G Wagen,0,75,169,0,15,19,4,54
1,BMW,G8X M4,0,66,186,0,25,17,6,69
2,Mercedes-Benz,G Wagen,0,90,179,0,15,10,11,66
3,Porsche,992 911,0,83,205,0,16,10,6,0
7,Rivian,R1S,0,40,200,0,15,10,17,54


## 4. Auction Buzzword Impact Analyzer
Uses TF-IDF and Ridge Regression to map specific words and bigrams directly to their dollar impact on the final sale price.

In [7]:
print("Vectorizing full text descriptions for Buzzword Analysis...")

# 1. Combine all text columns into a single blob
text_cols_all = [
    'Highlights', 'Equipment', 'Modifications', 'Known Flaws', 
    'Recent Service History', 'Ownership History', 'Seller Notes', 
    'Other Items Included in Sale'
]

df['buzzword_text_blob'] = df[text_cols_all].fillna('').astype(str).apply(lambda x: ' '.join(x), axis=1)

# 2. Build a dynamic exclusion list of Makes and Models
makes = df['Make'].dropna().str.lower().unique().tolist()

# Models are often multi-word (e.g. "911 Carrera"), so we split them into individual words
models_raw = df['Model'].dropna().str.lower().str.split().explode().unique().tolist()
models = [re.sub(r'[^a-z0-9]', '', str(m)) for m in models_raw if str(m).strip()]

# Add common brand jargon and standard auction boilerplate
extra_brands = ['amg', 'benz', 'carrera', 'm3', 'm4', 'boxster', 'cayman', 'cayenne', 'macan', 'bmw', 'porsche', 'mercedes']
auction_boilerplate = ['seller', 'reports', 'states', 'gallery', 'pictured', 'known', 'flaws', 'report', 'shows', 'notable', 'equipment', 'includes', 'included', 'sale', 'history', 'attached']

# Combine standard english stop words with our custom vehicle terms
custom_stop_words = list(TfidfVectorizer(stop_words='english').get_stop_words()) + makes + models + extra_brands + auction_boilerplate
custom_stop_words = list(set(custom_stop_words)) # Remove duplicates

# 3. Vectorize the text using TF-IDF (excluding the custom stop words)
tfidf_vectorizer_buzz = TfidfVectorizer(max_features=2500, stop_words=custom_stop_words, ngram_range=(1, 2))
tfidf_matrix_buzz = tfidf_vectorizer_buzz.fit_transform(df['buzzword_text_blob'])

# 4. Fit a Ridge regression model specifically to map word frequencies to dollar prices
print("Calculating financial impact of keywords using Ridge Regression...")
word_impact_model = Ridge(alpha=10.0) 
word_impact_model.fit(tfidf_matrix_buzz, df['Sold_Price'])

# 5. Extract the words, their calculated dollar impacts, and frequency
words = tfidf_vectorizer_buzz.get_feature_names_out()
impact_values = word_impact_model.coef_
frequencies = np.array(tfidf_matrix_buzz.sum(axis=0)).flatten()

# 6. Combine into a DataFrame
df_buzzwords = pd.DataFrame({
    'Word': words,
    'Frequency': frequencies,
    'Impact_Value': impact_values
})

# 7. Filter out extremely rare or irrelevant words to reduce noise
df_buzzwords = df_buzzwords[df_buzzwords['Frequency'] > 5.0]

display(df_buzzwords.sort_values(by="Impact_Value", ascending=False).head(10))
display(df_buzzwords.sort_values(by="Impact_Value", ascending=True).head(10))

Vectorizing full text descriptions for Buzzword Analysis...


/opt/conda/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['alfa', 'aston', 'bae', 'cars', 'company', 'cruise', 'general', 'jurassic', 'little', 'local', 'manx', 'martin', 'meyers', 'military', 'motors', 'roadsters', 'rolls', 'romeo', 'royce', 'stevenson', 'stewart', 'suspensions', 'systems', 'unlimited', 'vehicle'] not in stop_words.
  warnings.warn(


Calculating financial impact of keywords using Ridge Regression...


,Word,Frequency,Impact_Value
477,ceramic,190.645845,101355.571605
435,carbon,593.276772,98589.102497
2361,v12,94.414441,91407.519682
1266,lift,198.005595,67278.079137
2149,stitching,103.662178,66843.751749
969,forged,127.764413,63598.646024
2430,wheel steering,67.320938,60640.692200
1106,inch rear,149.885119,57519.915226
2360,v10,83.016420,53610.482086
293,axle,217.563223,47929.380249


,Word,Frequency,Impact_Value
1940,scratches,654.003795,-37544.333088
2410,wear,586.060432,-32820.774063
1267,lift kit,106.167212,-31730.930339
1754,rated horsepower,214.001890,-31311.871558
1137,inline,304.705005,-27809.264329
669,cylinder engine,150.417151,-27515.205117
521,chips,515.713139,-27470.215179
543,cloth,407.222109,-26235.237836
282,autocheck,174.196262,-25656.742862
1928,rust,318.460268,-25043.083826


## Export Results
Drops the massive raw text columns and saves the lightweight NLP features into the frontend data directory for quick loading in Streamlit.


In [8]:
OUTPUT_BRANDS = "../data/frontend_data/nlp_brands.csv"
OUTPUT_ARCHETYPES = "../data/frontend_data/nlp_archetypes.csv"
OUTPUT_EFFORT = "../data/frontend_data/nlp_effort_scores.csv"
OUTPUT_BUZZWORDS = "../data/frontend_data/nlp_buzzwords.csv" # NEW EXPORT PATH

# 1. Export Brands
brands_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Has_Premium_Mods', 'Premium_Brand_Count', 'Extracted_Brands_List']
if set(brands_cols).issubset(df.columns):
    df[brands_cols].to_csv(OUTPUT_BRANDS, index=False)
    print(f"Saved Brands to {OUTPUT_BRANDS}")

# 2. Export Archetypes
archetypes_cols = ['URL', 'Make', 'Model', 'Sold_Price', 'Archetype_Cluster']
if set(archetypes_cols).issubset(df.columns):
    df[archetypes_cols].to_csv(OUTPUT_ARCHETYPES, index=False)
    print(f"Saved Archetypes to {OUTPUT_ARCHETYPES}")

# 3. Export Section-Specific Length Scores
effort_cols = [
    'URL', 'Make', 'Model', 'Sold_Price', 
    'Modifications_WC', 'Equipment_WC', 'Highlights_WC', 
    'Known_Flaws_WC', 'Recent_Service_WC', 'Ownership_WC', 
    'Other_Items_WC', 'Seller_Notes_WC'
]
if set(effort_cols).issubset(df.columns):
    df[effort_cols].to_csv(OUTPUT_EFFORT, index=False)
    print(f"Saved Section Length Scores to {OUTPUT_EFFORT}")

# 4. Export Buzzwords
if 'df_buzzwords' in locals() and not df_buzzwords.empty:
    df_buzzwords.to_csv(OUTPUT_BUZZWORDS, index=False)
    print(f"Saved Buzzwords to {OUTPUT_BUZZWORDS}")

Saved Brands to ../data/frontend_data/nlp_brands.csv
Saved Archetypes to ../data/frontend_data/nlp_archetypes.csv
Saved Section Length Scores to ../data/frontend_data/nlp_effort_scores.csv
Saved Buzzwords to ../data/frontend_data/nlp_buzzwords.csv
